# HW2: Transformers

## Setup Instructions

1. **Make a personal copy of this notebook.** You will not be able to modify the master version of this assignment.
1. **Setup your GPU runtime**: Go to Runtime > Change Runtime Type and select "T4 GPU". Click "Save".
1. **Upload the Pile data**: On the "Files" tab, upload the `pile` data folder. You can also upload the `pile.zip` directly and use the provided cell to unzip things into the right place. The `pile/` folder should contain train.txt, dev.txt, and test.txt files. The code expects this folder to be in `content/pile/`, where `/content/` is the default home folder for Colab projects.

## TODOs

- You must first implement the `Transformer` and `MultiHeadTransformer` classes. After you've completed these classes and finished the notebook, you will copy and paste this cell into a Python script called `transformer.py`, and upload this to the Gradescope autograder.
- You must later implement the `generate` function; this should be an implementation of temperature sampling.
- After you've completed the full notebook, you will upload both `transformer.py` and your completed notebook (an `.ipynb` file) to Gradescope.

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Environment Setup

These cells download the dependencies you'll need, and also set up any needed environment variables.

In [9]:
# Install required packages
!pip install torch transformers datasets -q

# Verify installation
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.64 GB


In [10]:
# Upload a `pile.zip` folder to the `Files` tab. This cell will unzip
# that archive and put everything in the right place.
import os
import zipfile

PILE_DATA_ROOT = '/content/pile/'

if os.path.exists('/content/pile.zip'):
    with zipfile.ZipFile('pile.zip', 'r') as zip_ref:
        zip_ref.extractall('/content')
    PILE_DATA_ROOT = '/content/pile'
    print(f"Extracted pile data to {PILE_DATA_ROOT}")

# Data Loading Utilities

These functions load data from the Pile or from a small Shakespeare corpus (from HuggingFace).

In [11]:
import torch
from pathlib import Path
from transformers import AutoTokenizer
from datasets import load_dataset

def get_tokenizer():
    # We'll use the GPT-2 tokenizer.
    _tokenizer = AutoTokenizer.from_pretrained("gpt2")
    return _tokenizer

def load_pile_data(split, max_len=100):
    print(f"Loading Pile {split} data...")

    file_path = Path(PILE_DATA_ROOT) / f"{split}.txt"
    if not file_path.exists():
        raise FileNotFoundError(f"Cannot find file: {file_path}")

    with open(file_path, "r", encoding="utf-8") as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    tokenizer = get_tokenizer()
    rows = []
    for line in lines:
        ids = tokenizer.encode(line)[:max_len]
        if len(ids) < max_len:
            ids += [tokenizer.eos_token_id] * (max_len - len(ids))
        rows.append(ids)

    return torch.tensor(rows, dtype=torch.long)

def load_shakespeare_data(split='train', max_len=100, max_samples=None):
    print(f"Loading Shakespeare {split} data from HuggingFace...")
    dataset = load_dataset("2nji/Shakespeare_Corpus", split=split)

    # Tokenize
    tokenizer = get_tokenizer()

    # Extract text and tokenize
    rows = []
    for i, item in enumerate(dataset):
        if max_samples and i >= max_samples:
            break

        # Get text from the dataset (adjust key if needed - might be 'text', 'content', etc.)
        text = item.get('text', '') or item.get('content', '') or str(item)

        # Tokenize and truncate to max_len
        ids = tokenizer.encode(text)[:max_len]
        if len(ids) < max_len:
            ids += [tokenizer.eos_token_id] * (max_len - len(ids))
        rows.append(ids)

    print(f"Loaded {len(rows)} Shakespeare {split} tokens")
    return torch.tensor(rows, dtype=torch.long)

def load_shakespeare_data_hf(split='train', max_samples=None):
    print(f"Loading Shakespeare {split} data for HuggingFace...")

    # Load the dataset with the correct split
    dataset = load_dataset("2nji/Shakespeare_Corpus", split=split)

    # Extract text from each item in the dataset
    rows = []
    for i, item in enumerate(dataset):
        if max_samples and i >= max_samples:
            break

        # Handle different possible text field names
        text = item.get('text', '') or item.get('content', '') or str(item)
        # Split into lines if needed and filter empty lines
        rows.append(text)

    print(f"Loaded {len(rows)} Shakespeare {split} lines")
    return [{"text": row} for row in rows]

# Transformer Implementation

This is your main TODO. You will implement the `Transformer` and `MultiHeadTransformer` classes in the following cell.

You can try implementing `Transformer` first and testing it in the following cells before coming back to implement `MultiHeadTransformer`.

After you've completed the notebook and answered all questions in the written report, copy the contents of this cell and paste it into a script called `transformer.py`. Upload this script, as well as your .ipynb notebook, to HW2's code portal on Gradescope. (You can download your notebook by clicking File > Download > Download .ipynb).

In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class Transformer(nn.Module):
    """Multi-layer single-head Transformer decoder."""

    def __init__(self, vocab_size, hidden_dim, context_len, num_layers=2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.context_len = context_len
        self.num_layers = num_layers

        # 1. Token embeddings
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        # Positional embeddings
        self.pos_embedding = nn.Embedding(context_len, hidden_dim)

        # Create layers dynamically
        self.layers = nn.ModuleList()
        for _ in range(num_layers):
            # TODO: Add the query, key, value, and output projection matrices
            # (one per layer). These should map from a matrix of size `hidden_dim` to a matrix
            # of size `hidden_dim`. Also add the `W_up` and `W_down` matrices (one
            # per layer).
            # STUDENT START -----------------------------------------
            # Define matrices for one layer
            layer_modules = nn.ModuleDict({
                # Q, K, V projections
                'w_q': nn.Linear(hidden_dim, hidden_dim, bias=False),
                'w_k': nn.Linear(hidden_dim, hidden_dim, bias=False),
                'w_v': nn.Linear(hidden_dim, hidden_dim, bias=False),

                # Output projection
                'w_o': nn.Linear(hidden_dim, hidden_dim, bias=False),

                # MLP: Up and Down projections
                'w_up': nn.Linear(hidden_dim, 4 * hidden_dim), # Usually expand by 4x
                'w_down': nn.Linear(4 * hidden_dim, hidden_dim)
            })
            self.layers.append(layer_modules)
            # STUDENT END --------------------------------------------

        # Layer norm parameters for each layer
        self.gamma_attn = nn.ParameterList([nn.Parameter(torch.ones(hidden_dim)) for _ in range(num_layers)])
        self.beta_attn = nn.ParameterList([nn.Parameter(torch.zeros(hidden_dim)) for _ in range(num_layers)])
        self.gamma_mlp = nn.ParameterList([nn.Parameter(torch.ones(hidden_dim)) for _ in range(num_layers)])
        self.beta_mlp = nn.ParameterList([nn.Parameter(torch.zeros(hidden_dim)) for _ in range(num_layers)])

    def layer_norm(self, x, gamma, beta):
        # TODO: implement layer norm.
        # For now, this just returns x. Delete this line,
        # and replace it with this functionality:
        # x_hat = gamma * (x - mu) / sigma + beta
        # STUDENT START ------------------------------
        # Comment this out and replace it with your
        # LayerNorm implementation.

        # USE_LN = False # Setting False for Q1, True for Q2 -> FLAG for Q1 to write in document

        # if not USE_LN:
        #     return x

        # Calculate mean and variance along the last dimension
        mu = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)

        # Add epsilon for numerical stability (e.g., 1e-5)
        sigma = torch.sqrt(var + 1e-5)

        # Apply normalization: gamma * (x - mu) / sigma + beta
        return gamma * (x - mu) / sigma + beta

        # STUDENT END --------------------------------

    def forward(self, x):
        B, T = x.size()  # (Batch size, sequence length)

        # 1. Token embeddings + positional encodings
        positions = torch.arange(0, T, device=x.device).unsqueeze(0)
        h = self.embedding(x) + self.pos_embedding(positions)

        # Process through each layer
        for layer_idx in range(self.num_layers):
            layer = self.layers[layer_idx]
            residual = h  # Put the output of each layer in `h`

            # TODO: 2. Implement self-attention. Q, K, and V should
            # be of dimension (B, T). Recall that you must implement
            # a causal mask; the upper triangle of the attention scores must
            # be set to negative infinity before the softmax such that the model
            # cannot cheat by looking ahead beyond the current position.
            # Also remember to apply LayerNorm after the self-attention.
            # STUDENT START --------------------------
            # i. Project X into Q, K, and V matrices
            # All shapes: (B, T, hidden_dim)
            Q = layer['w_q'](h)
            K = layer['w_k'](h)
            V = layer['w_v'](h)

            # ii. Compute attention scores
            # (B, T, H) @ (B, H, T) -> (B, T, T)
            # Scale by sqrt(d_k)
            d_k = self.hidden_dim
            wei = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)

            # iii. Causal masking
            # Create a lower triangular mask of ones (1s)
            tril = torch.tril(torch.ones(T, T, device=x.device))
            # Replace 0s with -inf so softmax zeroes them out
            wei = wei.masked_fill(tril == 0, float('-inf'))

            # iv. Softmax and multiply by Values
            attn_weights = F.softmax(wei, dim=-1) # (B, T, T)
            out = attn_weights @ V # (B, T, T) @ (B, T, H) -> (B, T, H)

            # v. Output projection
            Z = layer['w_o'](out)

            # vi. Residual and LayerNorm (Apply norm to the SUM)
            # Note: The homework equation (7) implies Z' = Z + X (residual first)
            # Then apply norm. The prompt says "Apply a layer norm to the output of the attention A"
            h = self.layer_norm(residual + Z, self.gamma_attn[layer_idx], self.beta_attn[layer_idx])
            # STUDENT END ------------------------------

            # TODO: 3. Implement the MLP: h = ReLU(aW_up + b1)W_down + b2
            # STUDENT START -------------------------
            residual_mlp = h
            # H = ReLU(ZW_up + b1)W_down + b2
            # Note: nn.Linear handles bias automatically if bias=True (default)
            mlp_out = layer['w_up'](h)
            mlp_out = F.relu(mlp_out)
            mlp_out = layer['w_down'](mlp_out)
            # STUDENT END ----------------------------

            # TODO: 4. Add residual & layer norm to get the layer output. Store
            # the output in the `h` variable.
            # The current `self.layer_norm` method simply returns its input without
            # actually normalizing them; you will need to implement `self.layer_norm`
            # for this to work properly.
            # STUDENT START ----------------------------
            h = self.layer_norm(residual_mlp + mlp_out, self.gamma_mlp[layer_idx], self.beta_mlp[layer_idx])
            # STUDENT END ------------------------------

        return h


class MultiHeadTransformer(Transformer):
    """Multi-layer multi-head Transformer decoder."""

    def __init__(self, vocab_size, hidden_dim, context_len, num_heads=4, num_layers=2):
        # __init__ is mostly the same as in `Transformer`.
        super().__init__(vocab_size, hidden_dim, context_len, num_layers)
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        assert hidden_dim % num_heads == 0, "Hidden dim must be divisible by num_heads"

    def forward(self, x):
        B, T = x.size()

        # Embeddings
        positions = torch.arange(0, T, device=x.device).unsqueeze(0)
        h = self.embedding(x) + self.pos_embedding(positions)

        # Process through each layer
        for layer_idx in range(self.num_layers):
            layer = self.layers[layer_idx]
            residual = h

            # TODO: Implement multi-head attention. This should be similar
            # to your forward function in `Transformer`, except Q, K, and V should
            # be of size (B, T, num_heads, head_dim); your previous implementation
            # had them at size (B, T). Remember to apply residual connections and
            # LayerNorm, and to put an MLP after the self-attention (with its own
            # residual connection and LayerNorm). Also remember the causal mask.
            # STUDENT START ------------------------------------
            # 1. Project Q, K, V (B, T, hidden_dim)
            # We use the same linear layers defined in the parent class
            q = layer['w_q'](h)
            k = layer['w_k'](h)
            v = layer['w_v'](h)

            # 2. Reshape for Multi-Head Attention
            # Split hidden_dim into (num_heads, head_dim)
            # Transpose to get (B, num_heads, T, head_dim) so we can matrix mult the T dimensions
            q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
            k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
            v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

            # 3. Compute Attention Scores
            # (B, nh, T, hd) @ (B, nh, hd, T) -> (B, nh, T, T)
            # Scale by sqrt(head_dim) per standard scaled dot-product attention
            d_k = self.head_dim
            scores = (q @ k.transpose(-2, -1)) / math.sqrt(d_k)

            # 4. Causal Masking
            # Create mask (T, T) and broadcast it
            tril = torch.tril(torch.ones(T, T, device=x.device))
            scores = scores.masked_fill(tril == 0, float('-inf'))

            # 5. Softmax and Value Aggregation
            attn_weights = F.softmax(scores, dim=-1)
            # (B, nh, T, T) @ (B, nh, T, hd) -> (B, nh, T, hd)
            context = attn_weights @ v

            # 6. Concatenate Heads
            # Transpose back to (B, T, nh, hd) then flatten to (B, T, hidden_dim)
            context = context.transpose(1, 2).contiguous().view(B, T, self.hidden_dim)

            # 7. Output Projection
            z = layer['w_o'](context)

            # 8. Residual Connection & Layer Norm (Attention Block)
            h = self.layer_norm(residual + z, self.gamma_attn[layer_idx], self.beta_attn[layer_idx])

            # --- MLP Block (Same as Single-Head) ---
            residual_mlp = h
            mlp_out = layer['w_up'](h)
            mlp_out = F.relu(mlp_out)
            mlp_out = layer['w_down'](mlp_out)

            # Residual Connection & Layer Norm (MLP Block)
            h = self.layer_norm(residual_mlp + mlp_out, self.gamma_mlp[layer_idx], self.beta_mlp[layer_idx])
            # STUDENT END --------------------------------------------------------

        return h

# Training and Evaluation Utilities

In [34]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

def _ensure_2d_batch(batch):
    """Ensure batch is 2D tensor."""
    if isinstance(batch, (list, tuple)):
        if len(batch) == 1 and torch.is_tensor(batch[0]):
            batch = batch[0]
        else:
            batch = torch.stack([b if torch.is_tensor(b) else torch.tensor(b) for b in batch], dim=0)

    if not torch.is_tensor(batch):
        batch = torch.tensor(batch)

    if batch.dim() == 1:
        batch = batch.unsqueeze(0)

    return batch

def train(model, train_data, dev_data, epochs=10, lr=5e-3, device='cpu', save_path='model.pt'):
    """Train a transformer model."""
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()
    model.to(device)
    model.train()

    print(f"Starting training on {device}...")

    train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
    dev_loader = DataLoader(dev_data, batch_size=16)

    best_perp = float('inf')
    prev_perp = float('inf')

    for epoch in range(epochs):
        total_loss = 0
        for i, batch in enumerate(train_loader):
            batch = _ensure_2d_batch(batch)
            if batch.size(1) < 2:
                continue

            inputs = batch[:, :-1].to(device)
            targets = batch[:, 1:].to(device)

            optimizer.zero_grad()
            output = model(inputs)
            logits = torch.matmul(output, model.embedding.weight.t())
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

            if i % 100 == 0:
                print(f"Epoch {epoch} Step {i} Loss: {loss.item():.4f}")

        val_perp = evaluate_perplexity(model, dev_loader, device)
        print(f"Epoch {epoch} Completed. Dev Perplexity: {val_perp:.4f}")

        if val_perp > prev_perp * 1.1 and epoch > 2:
            print(f"Early stopping at epoch {epoch} due to increasing perplexity.")
            break

        prev_perp = val_perp
        if val_perp < best_perp:
            best_perp = val_perp
            torch.save(model.state_dict(), save_path)
            print(f"Best model saved to {save_path} with perplexity {best_perp:.4f}")

    if best_perp == float('inf'):
        torch.save(model.state_dict(), save_path)
        print(f"Model saved to {save_path}")


def evaluate_perplexity(model, data, device='cpu'):
    """Evaluate perplexity on dataset."""
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0
    total_tokens = 0

    with torch.no_grad():
        for batch in data:
            batch = _ensure_2d_batch(batch)
            if batch.size(1) < 2:
                continue

            inputs = batch[:, :-1].to(device)
            targets = batch[:, 1:].to(device)

            output = model(inputs)
            logits = torch.matmul(output, model.embedding.weight.t())
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            total_loss += loss.item() * targets.numel()
            total_tokens += targets.numel()

    model.train()

    if total_tokens == 0:
        return float("inf")

    avg_loss = total_loss / total_tokens
    perplexity = torch.exp(torch.tensor(avg_loss))
    return perplexity.item()

# Training from Scratch on the Pile

Train your transformer on general internet text (The Pile).

In [35]:
# Load The Pile
print("Loading The Pile...")
train_data = load_pile_data('train')
dev_data = load_pile_data('dev')
tokenizer = get_tokenizer()

print(f"Train data shape: {train_data.shape}")
print(f"Dev data shape: {dev_data.shape}")
print(f"Vocabulary size: {tokenizer.vocab_size}")

# Use GPU if available; else, use CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Loading The Pile...
Loading Pile train data...


Token indices sequence length is longer than the specified maximum sequence length for this model (1895 > 1024). Running this sequence through the model will result in indexing errors


Loading Pile dev data...


Token indices sequence length is longer than the specified maximum sequence length for this model (2018 > 1024). Running this sequence through the model will result in indexing errors


Train data shape: torch.Size([10000, 100])
Dev data shape: torch.Size([1000, 100])
Vocabulary size: 50257
Using device: cuda


In [36]:
# Initialize single-head transformer
vocab_size = tokenizer.vocab_size
hidden_dim = 256
context_len = 128

model_single = Transformer(vocab_size=vocab_size, hidden_dim=hidden_dim, context_len=context_len)
print(f"Model parameters: {sum(p.numel() for p in model_single.parameters()):,}")

# Train model. These are typical pre-training hyperparameters; you can feel
# free to modify the number of epochs and learning rate.
train(
    model_single,
    train_data,
    dev_data,
    epochs=5,  # Adjust as needed
    lr=5e-3,
    device=device,
    save_path='TRANSFORMER_pile.pt'
)

# Evaluate
perplexity = evaluate_perplexity(model_single, dev_data, device)
print(f"\nFinal Pile Dev Perplexity: {perplexity:.4f}")

Model parameters: 14,476,032
Starting training on cuda...
Epoch 0 Step 0 Loss: 168.8540
Epoch 0 Step 100 Loss: 10.1035
Epoch 0 Step 200 Loss: 8.5665
Epoch 0 Step 300 Loss: 8.3734
Epoch 0 Step 400 Loss: 8.4044
Epoch 0 Step 500 Loss: 8.3365
Epoch 0 Step 600 Loss: 8.4928
Epoch 0 Completed. Dev Perplexity: 3463.9194
Best model saved to TRANSFORMER_pile.pt with perplexity 3463.9194
Epoch 1 Step 0 Loss: 8.3126
Epoch 1 Step 100 Loss: 8.0780
Epoch 1 Step 200 Loss: 8.1669
Epoch 1 Step 300 Loss: 7.6311
Epoch 1 Step 400 Loss: 8.3152
Epoch 1 Step 500 Loss: 7.9666
Epoch 1 Step 600 Loss: 8.0392
Epoch 1 Completed. Dev Perplexity: 2922.6167
Best model saved to TRANSFORMER_pile.pt with perplexity 2922.6167
Epoch 2 Step 0 Loss: 8.0383
Epoch 2 Step 100 Loss: 8.1863
Epoch 2 Step 200 Loss: 8.0217
Epoch 2 Step 300 Loss: 7.8795
Epoch 2 Step 400 Loss: 8.2348
Epoch 2 Step 500 Loss: 7.9014
Epoch 2 Step 600 Loss: 8.0340
Epoch 2 Completed. Dev Perplexity: 3073.1150
Epoch 3 Step 0 Loss: 7.8469
Epoch 3 Step 100 Los

In [16]:
# Initialize multi-head transformer
model_pile = MultiHeadTransformer(
    vocab_size=vocab_size,
    hidden_dim=hidden_dim,
    context_len=context_len,
    num_heads=4
)
print(f"Model parameters: {sum(p.numel() for p in model_pile.parameters()):,}")

# Train model
train(
    model_pile,
    train_data,
    dev_data,
    epochs=10,  # Adjust as needed
    lr=5e-3,
    device=device,
    save_path='TRANSFORMER_MH_pile.pt'
)

# Evaluate
perplexity = evaluate_perplexity(model_pile, dev_data, device)
print(f"\nFinal Pile Dev Perplexity: {perplexity:.4f}")

Model parameters: 14,476,032
Starting training on cuda...
Epoch 0 Step 0 Loss: 152.1215
Epoch 0 Step 100 Loss: 10.1247
Epoch 0 Step 200 Loss: 8.5421
Epoch 0 Step 300 Loss: 8.3929
Epoch 0 Step 400 Loss: 8.3680
Epoch 0 Step 500 Loss: 7.6993
Epoch 0 Step 600 Loss: 8.1757
Epoch 0 Completed. Dev Perplexity: 2671.6230
Best model saved to TRANSFORMER_MH_pile.pt with perplexity 2671.6230
Epoch 1 Step 0 Loss: 7.6690
Epoch 1 Step 100 Loss: 8.2506
Epoch 1 Step 200 Loss: 7.7088
Epoch 1 Step 300 Loss: 7.5921
Epoch 1 Step 400 Loss: 7.9403
Epoch 1 Step 500 Loss: 7.9130
Epoch 1 Step 600 Loss: 7.5720
Epoch 1 Completed. Dev Perplexity: 2544.6121
Best model saved to TRANSFORMER_MH_pile.pt with perplexity 2544.6121
Epoch 2 Step 0 Loss: 7.8874
Epoch 2 Step 100 Loss: 7.9460
Epoch 2 Step 200 Loss: 7.5460
Epoch 2 Step 300 Loss: 8.2227
Epoch 2 Step 400 Loss: 7.8664
Epoch 2 Step 500 Loss: 7.6369
Epoch 2 Step 600 Loss: 7.8604
Epoch 2 Completed. Dev Perplexity: 2535.7493
Best model saved to TRANSFORMER_MH_pile.pt

# Text Generation

This is your second main TODO. In the following cell, you will implement temperature sampling.

After you do this, you can run the following cell to generate from your language model. Don't worry if these outputs look a bit ungrammatical and weird. It shouldn't be entirely random tokens—but it will be close (at least with the default hyperparameters)!

Optionally, for extra credit, you can implement nucleus sampling in the subsequent cell. If you do this, be sure to fill out the written report telling us you did this, and giving some example generations.

In [17]:
import torch
import torch.nn as nn

def generate(model, tokenizer, start_text, max_new_tokens=30, device='cpu', temperature=0.8):
    model.eval()
    encoded = tokenizer.encode(start_text)
    tokens = torch.tensor(encoded, dtype=torch.long).unsqueeze(0).to(device)  # (1, T)

    eos_id = getattr(tokenizer, "eos_token_id", None)
    context_len = getattr(model, "context_len", None)

    for _ in range(max_new_tokens):
        with torch.no_grad():
            model_input = tokens
            if context_len is not None and model_input.size(1) > context_len:
                model_input = model_input[:, -context_len:]  # keep only latest context

            output = model(model_input)  # (1, T, H)
            logits = torch.matmul(output, model.embedding.weight.t())  # (1, T, Vocab)

            # TODO: Implement temperature sampling to get `next_token`. Recall that this involves
            # dividing the logits by the temperature before taking their softmax.
            # You should then sample from the resulting probability distribution; use
            # `torch.multinomial` with `num_samples=1` to sample one token from the
            # probability distribution.
            # STUDENT START --------------------------
            # 1. Get the logits for the last token only
            # logits shape: (Batch, Sequence, Vocab) -> We want (Batch, Vocab)
            next_token_logits = logits[:, -1, :]

            # 2. Apply Temperature Scaling
            # Divide logits by temperature
            next_token_logits = next_token_logits / temperature

            # 3. Compute Probabilities
            # Apply softmax to get valid probabilities summing to 1
            probs = F.softmax(next_token_logits, dim=-1)

            # 4. Sample from the Distribution
            # multinomial samples indices based on the probability mass
            next_token = torch.multinomial(probs, num_samples=1)
            # raise NotImplementedError("Temperature sampling has not yet been implemented.")
            # STUDENT END ----------------------------

            tokens = torch.cat((tokens, next_token), dim=1)

            if eos_id is not None and next_token.item() == eos_id:
                break

    decoded = tokenizer.decode(tokens[0].tolist())
    return decoded

In [18]:
import torch
import torch.nn as nn

def nucleus_sampling_generate(model, tokenizer, start_text, max_new_tokens=30, device='cpu', p=0.9):
    model.eval()
    encoded = tokenizer.encode(start_text)
    tokens = torch.tensor(encoded, dtype=torch.long).unsqueeze(0).to(device)  # (1, T)

    eos_id = getattr(tokenizer, "eos_token_id", None)
    context_len = getattr(model, "context_len", None)

    for _ in range(max_new_tokens):
        with torch.no_grad():
            model_input = tokens
            if context_len is not None and model_input.size(1) > context_len:
                model_input = model_input[:, -context_len:]  # keep only latest context

            output = model(model_input)  # (1, T, H)
            logits = torch.matmul(output, model.embedding.weight.t())  # (1, T, Vocab)

            # TODO (extra credit): Implement nucleus sampling to get `next_token`.
            # Recall that this involves clipping the distribution as soon as we
            # have a collection of tokens exceeding cumulative probability `p`.
            # You should then sample from the resulting probability distribution; use
            # `torch.multinomial` with `num_samples=1` to sample one token from the
            # probability distribution.
            # STUDENT START --------------------------
            # 1. Sort logits descending
            sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
            sorted_probs = F.softmax(sorted_logits, dim=-1)

            # 2. Compute cumulative probabilities
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

            # 3. Remove tokens with cumulative probability above the threshold (p)
            # Shift the mask right to keep the first token above the threshold
            sorted_indices_to_remove = cumulative_probs > p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0

            # 4. Scatter sorted mask back to original indices
            indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)

            # 5. Set removed logits to -inf
            next_token_logits = next_token_logits.masked_fill(indices_to_remove, float('-inf'))

            # 6. Sample
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            # raise NotImplementedError("Nucleus sampling has not yet been implemented.")
            # STUDENT END ----------------------------

            tokens = torch.cat((tokens, next_token), dim=1)

            if eos_id is not None and next_token.item() == eos_id:
                break

    decoded = tokenizer.decode(tokens[0].tolist())
    return decoded

In [19]:
# Test generation with your trained model
prompts = [
    "The quick brown fox",
    "In the beginning",
    "What is 5 + 5?",
    "The meaning of life is"
]

print("Generating text samples with PILE model...\n")
for prompt in prompts:
    try:
        generated = generate(model_pile, tokenizer, prompt, max_new_tokens=30, device=device)
        print(f"Prompt: {prompt}")
        print(f"Generated: {generated}")
        print("-" * 80)
    except NotImplementedError:
        print("⚠️  Please implement the generate() function first!")
        break

Generating text samples with PILE model...

Prompt: The quick brown fox
Generated: The quick brown fox Kab is creda the the of examined a the a Sinclair the with Type a be the the the second was the two Registration with be the the a
--------------------------------------------------------------------------------
Prompt: In the beginning
Generated: In the beginning the to assumed is activity the to a known and ne of is the as the blood ch by a AN) a the conducted is hair the is they
--------------------------------------------------------------------------------
Prompt: What is 5 + 5?
Generated: What is 5 + 5? of a the the age the a low to the the is the the a- The the the a where a a if the calculating using the and a
--------------------------------------------------------------------------------
Prompt: The meaning of life is
Generated: The meaning of life is the India and the is the the their clinical year in the to a be a not�,'ve metric was matt con of extensively of was thej
---

# Training and Fine-tuning on Shakespeare

In this section, you will train a series of Transformers on a small Shakespeare corpus. You will train a Transformer from scratch on this data, and compare this model to your pre-trained Pile model fine-tuned on the Shakespeare corpus. Finally, you will fine-tune GPT-2 (a model pre-trained on far more data than we have time for) on the Shakespeare corpus.

## Load Shakespeare Data

In [20]:
# Load Shakespeare dataset from HuggingFace
# https://huggingface.co/datasets/2nji/Shakespeare_Corpus/
print("Loading Shakespeare dataset...")
train_data_shakes = load_shakespeare_data('train')
dev_data_shakes = load_shakespeare_data('validation')

print(f"Shakespeare train data shape: {train_data_shakes.shape}")
print(f"Shakespeare dev data shape: {dev_data_shakes.shape}")

Loading Shakespeare dataset...
Loading Shakespeare train data from HuggingFace...


README.md:   0%|          | 0.00/538 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/513k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/158k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/128k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4621 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1445 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1156 [00:00<?, ? examples/s]

Loaded 4621 Shakespeare train tokens
Loading Shakespeare validation data from HuggingFace...
Loaded 1156 Shakespeare validation tokens
Shakespeare train data shape: torch.Size([4621, 100])
Shakespeare dev data shape: torch.Size([1156, 100])


## Baseline: Evaluate Pile Model on Shakespeare (Zero-shot)

Before training from scratch, see how well your model pre-trained on the Pile does on Shakespeare without any adaptation.

In [21]:
pile_on_shakespeare_perp = evaluate_perplexity(model_pile, dev_data_shakes, device)
print(f"\Pile model on Shakespeare (zero-shot): {pile_on_shakespeare_perp:.4f}")
print("This is a baseline - fine-tuning should improve this!")

<>:2: SyntaxWarning: invalid escape sequence '\P'
<>:2: SyntaxWarning: invalid escape sequence '\P'
/tmp/ipython-input-658/4015768402.py:2: SyntaxWarning: invalid escape sequence '\P'
  print(f"\Pile model on Shakespeare (zero-shot): {pile_on_shakespeare_perp:.4f}")


\Pile model on Shakespeare (zero-shot): 16.6351
This is a baseline - fine-tuning should improve this!


In [22]:
# Train from scratch
model_shakespeare = MultiHeadTransformer(
    vocab_size=vocab_size,
    hidden_dim=hidden_dim,
    context_len=context_len,
    num_heads=4
)
model_shakespeare.to(device)

print("Training model on Shakespeare...")

train(
    model_shakespeare,
    train_data_shakes,
    dev_data_shakes,
    epochs=10,
    lr=5e-3,
    device=device,
    save_path='TRANSFORMER_MH_shakespeare.pt'
)

# Evaluate
shakespeare_perp = evaluate_perplexity(model_shakespeare, dev_data_shakes, device)
print(f"\nDev perplexity: {shakespeare_perp:.4f}")

Training model on Shakespeare...
Starting training on cuda...
Epoch 0 Step 0 Loss: 54.3816
Epoch 0 Step 100 Loss: 2.6026
Epoch 0 Step 200 Loss: 2.2985
Epoch 0 Completed. Dev Perplexity: 11.6720
Best model saved to TRANSFORMER_MH_shakespeare.pt with perplexity 11.6720
Epoch 1 Step 0 Loss: 2.0861
Epoch 1 Step 100 Loss: 1.7573
Epoch 1 Step 200 Loss: 2.6899
Epoch 1 Completed. Dev Perplexity: 9.5034
Best model saved to TRANSFORMER_MH_shakespeare.pt with perplexity 9.5034
Epoch 2 Step 0 Loss: 3.3198
Epoch 2 Step 100 Loss: 2.1394
Epoch 2 Step 200 Loss: 1.4563
Epoch 2 Completed. Dev Perplexity: 8.5262
Best model saved to TRANSFORMER_MH_shakespeare.pt with perplexity 8.5262
Epoch 3 Step 0 Loss: 1.9563
Epoch 3 Step 100 Loss: 2.3032
Epoch 3 Step 200 Loss: 2.0150
Epoch 3 Completed. Dev Perplexity: 8.2968
Best model saved to TRANSFORMER_MH_shakespeare.pt with perplexity 8.2968
Epoch 4 Step 0 Loss: 2.1198
Epoch 4 Step 100 Loss: 1.6046
Epoch 4 Step 200 Loss: 1.9230
Epoch 4 Completed. Dev Perplexity: 

## Fine-tune on Shakespeare

In [23]:
# Load your pre-trained Pile model
model_pile_shakespeare = MultiHeadTransformer(
    vocab_size=vocab_size,
    hidden_dim=hidden_dim,
    context_len=context_len,
    num_heads=4
)
model_shakespeare.load_state_dict(torch.load('TRANSFORMER_MH_pile.pt', map_location=device))
model_shakespeare.to(device)

print("Fine-tuning PILE model on Shakespeare...")

# Fine-tune on Shakespeare (use lower learning rates and num epochs for fine-tuning)
train(
    model_shakespeare,
    train_data_shakes,
    dev_data_shakes,
    epochs=5,
    lr=5e-4,
    device=device,
    save_path='TRANSFORMER_MH_pile_shakespeare.pt'
)

# Evaluate
shakespeare_perp = evaluate_perplexity(model_shakespeare, dev_data_shakes, device)
print(f"\nShakespeare-adapted model perplexity: {shakespeare_perp:.4f}")

Fine-tuning PILE model on Shakespeare...
Starting training on cuda...
Epoch 0 Step 0 Loss: 3.5623
Epoch 0 Step 100 Loss: 2.1684
Epoch 0 Step 200 Loss: 2.2771
Epoch 0 Completed. Dev Perplexity: 8.9721
Best model saved to TRANSFORMER_MH_pile_shakespeare.pt with perplexity 8.9721
Epoch 1 Step 0 Loss: 2.1714
Epoch 1 Step 100 Loss: 1.5650
Epoch 1 Step 200 Loss: 2.5674
Epoch 1 Completed. Dev Perplexity: 8.5035
Best model saved to TRANSFORMER_MH_pile_shakespeare.pt with perplexity 8.5035
Epoch 2 Step 0 Loss: 1.4116
Epoch 2 Step 100 Loss: 2.6089
Epoch 2 Step 200 Loss: 2.4786
Epoch 2 Completed. Dev Perplexity: 8.2191
Best model saved to TRANSFORMER_MH_pile_shakespeare.pt with perplexity 8.2191
Epoch 3 Step 0 Loss: 1.8150
Epoch 3 Step 100 Loss: 1.9804
Epoch 3 Step 200 Loss: 1.9399
Epoch 3 Completed. Dev Perplexity: 8.0208
Best model saved to TRANSFORMER_MH_pile_shakespeare.pt with perplexity 8.0208
Epoch 4 Step 0 Loss: 1.2390
Epoch 4 Step 100 Loss: 1.6658
Epoch 4 Step 200 Loss: 1.5670
Epoch 4 Co

In [24]:
# TODO: Re-evaluate your fine-tuned model's perplexity
# on the dev split of the Pile.
# STUDENT START ----------------------------
print("Evaluating fine-tuned model on original Pile data...")
pile_perp_finetuned = evaluate_perplexity(model_shakespeare, dev_data, device)
print(f"Fine-tuned model perplexity on original Pile data: {pile_perp_finetuned:.4f}")
# STUDENT END ------------------------------

Evaluating fine-tuned model on original Pile data...
Fine-tuned model perplexity on original Pile data: 12602.8594


In [25]:
# Generate Shakespearean text
shakespeare_prompts = [
    "To be or not to be",
    "O Romeo, Romeo",
    "Friends, Romans, countrymen",
    "All the world's a stage"
]
prompts = [
    "The quick brown fox",
    "In the beginning",
    "What is 5 + 5?",
    "The meaning of life is"
]

model_pile_shakespeare.to(device)

for prompt in shakespeare_prompts:
    try:
        generated = generate(model_pile_shakespeare, tokenizer, prompt, max_new_tokens=50, device=device, temperature=0.9)
        print(f"Prompt: {prompt}")
        print(f"Generated: {generated}")
        print("-" * 80)
    except NotImplementedError:
        print("Please implement the generate() function first!")
        break

for prompt in prompts:
    try:
        generated = generate(model_pile_shakespeare, tokenizer, prompt, max_new_tokens=50, device=device, temperature=0.9)
        print(f"Prompt: {prompt}")
        print(f"Generated: {generated}")
        print("-" * 80)
    except NotImplementedError:
        print("Please implement the generate() function first!")
        break

Prompt: To be or not to be
Generated: To be or not to be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be be
--------------------------------------------------------------------------------
Prompt: O Romeo, Romeo
Generated: O Romeo, Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo Romeo
--------------------------------------------------------------------------------
Prompt: Friends, Romans, countrymen
Generated: Friends, Romans, countrymenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmenmen
---------------------------------------------------------

# Fine-tune GPT-2 on Shakespeare

Compare your transformer with a pre-trained model. How much does the much larger scale of GPT-2's training data help?

## Fine-tune GPT-2

In [26]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset
import os

# Load GPT-2
MODEL_NAME = "gpt2"
print(f"Loading {MODEL_NAME}...")
tokenizer_hf = AutoTokenizer.from_pretrained(MODEL_NAME)
model_hf = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

if tokenizer_hf.pad_token is None:
    tokenizer_hf.pad_token = tokenizer_hf.eos_token

# Load Shakespeare dataset
print("Loading Shakespeare data...")
train_data_hf = load_shakespeare_data_hf('train')
dev_data_hf = load_shakespeare_data_hf('validation')

# Convert to HuggingFace Dataset format
train_dataset = Dataset.from_list(train_data_hf)
dev_dataset = Dataset.from_list(dev_data_hf)

# Tokenize
def preprocess_function(examples):
    return tokenizer_hf(examples['text'], truncation=True, padding="max_length", max_length=128)

train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=['text'])
dev_dataset = dev_dataset.map(preprocess_function, batched=True, remove_columns=['text'])

# TODO: play around with these hyperparameters, and see how well you can
# optimize GPT-2's dev perplexity on Shakespeare data.
training_args = TrainingArguments(
    output_dir="./gpt2_shakespeare/",
    learning_rate=5e-4,     # Play around with this hyperparameter
    num_train_epochs=3,     # Play around with this hyperparameter
    per_device_train_batch_size=4,
    save_steps=500,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=100,
    save_total_limit=2,
    report_to="none",
)

trainer = Trainer(
    model=model_hf,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer_hf, mlm=False),
)

print("Starting GPT-2 fine-tuning...")
trainer.train()

print("Saving model...")
model_hf.save_pretrained("./gpt2_shakespeare/")
tokenizer_hf.save_pretrained("./gpt2_shakespeare/")
print("\nGPT-2 fine-tuning complete.")

Loading gpt2...


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saving model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


GPT-2 fine-tuning complete.


## Evaluate GPT-2

Compare GPT-2's perplexity with your custom transformer. These results may not be what you expect—but hold off your final judgment of quality until you see the generations.

In [27]:
eval_results = trainer.evaluate()
gpt2_loss = eval_results['eval_loss']
print(gpt2_loss)
gpt2_perplexity = torch.exp(torch.tensor(gpt2_loss)).item()

print(f"GPT-2 (Shakespeare fine-tuned):              {gpt2_perplexity:.4f}")

4.333057880401611
GPT-2 (Shakespeare fine-tuned):              76.1769


## Generate with GPT-2

In [28]:
# Generate Shakespeare with GPT-2
shakespeare_prompts = [
    "To be or not to be",
    "O Romeo, Romeo",
    "Friends, Romans, countrymen",
    "All the world's a stage"
]
prompts = [
    "The quick brown fox",
    "In the beginning",
    "What is 5 + 5?",
    "The meaning of life is"
]

model_hf.eval()
for prompt in shakespeare_prompts:
    inputs = tokenizer_hf(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model_hf.generate(
            inputs.input_ids,
            max_new_tokens=50,
            temperature=0.9,
            do_sample=True,
            pad_token_id=tokenizer_hf.eos_token_id
        )

    generated = tokenizer_hf.decode(outputs[0], skip_special_tokens=True)
    print(f"Prompt: {prompt}")
    print(f"Generated: {generated}")
    print("-" * 80)
for prompt in prompts:
    inputs = tokenizer_hf(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model_hf.generate(
            inputs.input_ids,
            max_new_tokens=50,
            temperature=0.9,
            do_sample=True,
            pad_token_id=tokenizer_hf.eos_token_id
        )

    generated = tokenizer_hf.decode(outputs[0], skip_special_tokens=True)
    print(f"Prompt: {prompt}")
    print(f"Generated: {generated}")
    print("-" * 80)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Prompt: To be or not to be
Generated: To be or not to be. Take one step into some darkness of the night: I can tell you anything in the world, That is close to me: but that I'll give your worship, I do not mind to look upon. The morning is as the wild-cat
--------------------------------------------------------------------------------
Prompt: O Romeo, Romeo
Generated: O Romeo, Romeo, Romeo! Romeo! Romeo! Romeo! my lord! Romeo! why, is it he not Romeo? O Romeo! why, Romeo! why, Romeo! What is that? O Romeo! how did he not call for him? O O
--------------------------------------------------------------------------------
Prompt: Friends, Romans, countrymen
Generated: Friends, Romans, countrymen, and all, Showing their readiness to use us. See that the Roman camp Attend us at the cypress groan; And we, our friends, march on to the cypress. Go on, and we'll go on: Tell us
--------------------------------------------------------------------------------
Prompt: All the world's a stage
Generat